# clean_page — research notebook

The V1 tool. Runs between `load_memory` and `analyze_page`. The extension sends only the **URL**; the server fetches the page and strips boilerplate with [trafilatura](https://trafilatura.readthedocs.io/).

```
clean_page(url) -> CleanPageResult(ok, clean_text, title, word_count, truncated, error)
```

**Resilience rule: tools never raise.** Failures come back as `ok=False` with a user-safe `error`, and the graph routes to a graceful exit.

Trade-off accepted for V1: server fetch cannot see login-walled or heavily client-rendered pages; the sufficiency check in `analyze_page` catches those.

In [ ]:
# pip install -q trafilatura   (in research/requirements.txt)
import re
import trafilatura
from dataclasses import dataclass
from urllib.parse import urlparse

MAX_WORDS = 6000

@dataclass
class CleanPageResult:
    ok: bool
    clean_text: str = ""
    title: str = ""
    word_count: int = 0     # words after cleaning, before capping
    truncated: bool = False
    error: str = ""         # human-readable, safe to surface in the side panel

def clean_page(url: str) -> CleanPageResult:
    """Fetch a page and return clean, capped article text. Never raises."""
    try:
        parsed = urlparse(str(url))
        if parsed.scheme not in ("http", "https") or not parsed.netloc:
            return CleanPageResult(ok=False, error="Not a valid web address.")

        html = trafilatura.fetch_url(url)
        if not html:
            return CleanPageResult(ok=False, error="Could not fetch the page (offline, blocked, or requires login).")

        text = trafilatura.extract(html, url=url, include_comments=False,
                                   include_tables=True) or ""
        meta = trafilatura.extract_metadata(html)
        title = (meta.title if meta and meta.title else "").strip()

        text = re.sub(r"\s+", " ", text).strip()
        if not text:
            return CleanPageResult(ok=False, title=title,
                                   error="Fetched the page but found no readable article text.")

        words = text.split(" ")
        truncated = len(words) > MAX_WORDS
        return CleanPageResult(ok=True, clean_text=" ".join(words[:MAX_WORDS]),
                               title=title, word_count=len(words), truncated=truncated)
    except Exception as e:                       # noqa: BLE001 — resilience boundary
        return CleanPageResult(ok=False, error=f"Unexpected extraction failure: {e}")

## Try it

In [ ]:
r = clean_page("https://en.wikipedia.org/wiki/Photosynthesis")
print(r.ok, "|", r.title, "|", r.word_count, "words | truncated:", r.truncated)
print(r.clean_text[:300], "...")

In [ ]:
# sanity: content kept, failures graceful, never raised
assert r.ok and "chlorophyll" in r.clean_text.lower()

for bad_url in ["https://this-domain-does-not-exist-xyz123.invalid/page",
                "not even a url", "ftp://example.com/x", 12345]:
    res = clean_page(bad_url)
    assert res.ok is False and res.error, bad_url
    print("graceful:", res.error)
print("checks passed")

## Notes

- **Never raises.** The conditional edge after this node checks `ok`; on failure the run aborts gracefully and the SSE stream surfaces `error` to the side panel.
- **SSRF guard needed in production:** the server fetches user-supplied URLs, so before fetching, resolve the host and reject private and loopback ranges (127.0.0.0/8, 10/8, 172.16/12, 192.168/16, 169.254/16) and cloud metadata IPs. The scheme check above is only the first layer. Set an explicit fetch timeout.
- Deterministic, so it runs as a plain graph node; also registered with `@tool` for future agent use (fact-check flow).
- Fetch failure and empty-text are distinct errors so the UI can word them differently.
- Later: `favor_recall=True` may do better on doc-style sites; tune once real misextractions exist.